<a href="https://colab.research.google.com/github/arinamartynyuk/-4-9/blob/main/%D0%9B%D0%A0_6_%D0%9C%D0%B0%D1%80%D1%82%D0%B8%D0%BD%D1%8E%D0%BA4_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Лабораторна робота № 6 — Класифікація. Запобігання перенавчанню**

Датасет: Diabetes Prediction Dataset. Для запуску в Google Colab оберіть спосіб завантаження CSV у клітинці 3 та завантажте файл `diabetes_prediction_dataset.csv`. Код і висновки для основних завдань заповнені. Виконуйте клітинки послідовно.

Кореляції, IQR та асиметрія досліджуються лише на train. Test не балансується і не використовується для вибору параметрів.

In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

**Спосіб 1.** Завантажити CSV із комп’ютера (Google Colab)

In [ ]:
# Завантаження файлу
uploaded = files.upload()

In [ ]:
df = pd.read_csv('diabetes_prediction_dataset.csv')
df

**Спосіб 2.** Завантажити датасет з Kaggle (Google Colab)

In [ ]:
# Альтернативний спосіб завантаження через KaggleHub (необов'язковий):
# import kagglehub
# path = kagglehub.dataset_download('iammustafatz/diabetes-prediction-dataset')
# print(path)


Using Colab cache for faster access to the 'diabetes-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/diabetes-prediction-dataset


In [ ]:
# Для альтернативного способу KaggleHub:
# df = pd.read_csv(os.path.join(path, 'diabetes_prediction_dataset.csv'))
# df


Вивести інформацію про датасет

In [ ]:
df.info()
print('\nВисновок: числові та категоріальні типи даних відображені вище.')

Провести дослідження категоріальних ознак

In [ ]:
df['gender'].value_counts()

In [ ]:
df['smoking_history'].value_counts()

In [ ]:
df

Вивести типи даних

In [ ]:
df.dtypes

Дослідити цільову змінну: перевірити баланс класів

In [ ]:
df['diabetes'].value_counts()

Висновок: визначте за результатом, чи є дисбаланс класів.

Перевірити наявність пропущених даних


In [ ]:
df.isnull().sum()

Наявність дублікатів


In [ ]:
df.duplicated().sum()

Видалити дублікати

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df

Ще раз перевірити баланс цільової змінної

In [ ]:
df['diabetes'].value_counts()

Побудувати гістограму для цільової змінної


In [ ]:
sns.countplot(x='diabetes', data=df, color='tab:blue')
plt.title('Розподіл цільової змінної diabetes')
plt.show()

Переглянути описові статистики **навчальної вибірки**

In [ ]:
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [ ]:
X = df.drop(columns=['diabetes'])
y = df['diabetes']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

Закодувати категоріальні ознаки **після поділу**. Створюємо стовпці за категоріями з train; у test залишаємо точно такий самий набір стовпців.

In [ ]:
X_train = pd.get_dummies(X_train, columns=['gender', 'smoking_history'], drop_first=False)
X_test = pd.get_dummies(X_test, columns=['gender', 'smoking_history'], drop_first=False)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
print('Train columns:', X_train.shape[1], 'Test columns:', X_test.shape[1])

In [ ]:
X_train.describe()

Перевірити кореляцію ознак із цільовою змінною **лише на train**

In [ ]:
corr_target = X_train.corrwith(y_train).sort_values(key=lambda s: s.abs(), ascending=False)
corr_target

In [ ]:
corr_features = X_train.corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr_features, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Кореляція між ознаками (train)')
plt.tight_layout()
plt.show()

**Висновок:** які ознаки сильно корелюють між собою або з цільовою змінною? Чи потрібно видаляти ознаки? Обґрунтуйте відповідь. **Тестові дані для такого рішення не використовуємо.**

Перевірити викиди **лише на навчальній вибірці**

In [ ]:
continuous_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']
plt.figure(figsize=(12, 5))
X_train[continuous_cols].boxplot()
plt.title('Boxplot числових ознак (train)')
plt.xticks(rotation=20)
plt.show()

**Висновок:** викиди перевіряємо, але в цій лабораторній роботі автоматично **не видаляємо**. Екстремальні значення глюкози та HbA1c можуть бути клінічно важливими.

**Необов’язковий приклад.** Якщо потрібно видалити викиди методом IQR, межі обчислюємо **тільки на train**, видаляємо відповідні рядки з `X_train` і `y_train`, а `X_test` та `y_test` не змінюємо. У цій роботі приклад не запускаємо.

In [ ]:
# IQR для bmi можна обчислити на train так:
q1 = X_train['bmi'].quantile(0.25)
q3 = X_train['bmi'].quantile(0.75)
iqr = q3 - q1
lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('IQR:', iqr, 'Межі:', lower_bound, upper_bound)
# Рядки не видаляємо, щоб зберегти дані для навчання.

Перевірити асиметрію розподілів ознак **на train**

In [ ]:
X_train[continuous_cols].skew().sort_values(ascending=False)

In [ ]:
print('Пропуски train:\n', X_train.isnull().sum())
print('\nПропуски test:\n', X_test.isnull().sum())

**Проміжний висновок:** опишіть баланс класів, кореляції та викиди на train. Тестову вибірку залишаємо незмінною.

Масштабуємо ознаки: `fit_transform` лише для train, `transform` — для test.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Балансуємо лише навчальну вибірку за допомогою SMOTE

In [ ]:
print('До SMOTE:\n', y_train.value_counts())
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)
print('\nПісля SMOTE:\n', pd.Series(y_train_balanced).value_counts())

Оголошуємо моделі

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RidgeClassifier': RidgeClassifier(),
    'SGDClassifier': SGDClassifier(random_state=42, max_iter=1000),
    'SVC': SVC(random_state=42)
}
models

Навчити моделі, вивести матриці плутанини, класифікаційний звіт


In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train_balanced, y_train_balanced)
    y_pred = model.predict(X_test_scaled)
    results[name] = y_pred
    print(f'\n===== {name} =====')
    print(classification_report(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Матриця плутанини: {name}')
    plt.xlabel('Передбачений клас')
    plt.ylabel('Фактичний клас')
    plt.show()

Оптимізувати гіперпараметри


In [ ]:
param_grids = {
    'LogisticRegression': {'model__C': [0.1, 1, 10], 'model__solver': ['liblinear', 'lbfgs']},
    'RidgeClassifier': {'model__alpha': [0.1, 1.0, 10.0]},
    'SGDClassifier': {'model__loss': ['hinge', 'log_loss'], 'model__alpha': [0.0001, 0.001, 0.01]},
    'SVC': {'model__C': [0.1, 1, 10], 'model__kernel': ['linear', 'rbf']}
}
param_grids

Підбираємо параметри методом **HalvingGridSearchCV** із `cv=3`, `scoring="f1"`. Масштабування та SMOTE виконуються в кожному навчальному згині, щоб не було витоку даних. На малих порціях для пошуку задаємо достатній мінімальний розмір навчальних даних.

In [ ]:
# Для HalvingGridSearchCV потрібен цей імпорт перед основним імпортом.
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from imblearn.pipeline import Pipeline

In [ ]:
best_models = {}
best_scores = {}
for name, model in models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('model', model)])
    search = HalvingGridSearchCV(
        pipe, param_grids[name], cv=3, scoring='f1', min_resources=300,
        factor=3, n_jobs=-1, random_state=42, error_score='raise'
    )
    search.fit(X_train, y_train)
    best_models[name] = search.best_estimator_
    best_scores[name] = search.best_score_
    print(f'\n===== {name} =====')
    print('Найкращі параметри:', search.best_params_)
    print('Середній F1 CV:', search.best_score_)
    pred = search.best_estimator_.predict(X_test)
    print(classification_report(y_test, pred, zero_division=0))
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Матриця плутанини (оптимізована): {name}')
    plt.xlabel('Передбачений клас'); plt.ylabel('Фактичний клас')
    plt.show()

Вибрати найкращу модель за середнім F1 на кросвалідації (а не за тестовою вибіркою)

In [ ]:
best_name = max(best_scores, key=best_scores.get)
best_model = best_models[best_name]
print('Найкраща модель за CV F1:', best_name)
print('F1:', best_scores[best_name])

In [ ]:
rng = np.random.RandomState(42)
selected_positions = rng.choice(len(X_test), size=10, replace=False)
X_10 = X_test.iloc[selected_positions]
y_10 = y_test.iloc[selected_positions]
pred_10 = best_model.predict(X_10)
comparison = pd.DataFrame({
    'Індекс': X_10.index,
    'Істинний клас': y_10.values,
    'Передбачений клас': pred_10\})
comparison

**Підсумкові висновки (заповніть самостійно):**

1. Який дисбаланс класів спостерігався? Що показали кореляція та перевірка викидів **на train**?
2. Які моделі дали кращі `F1` і `recall` для класу `diabetes = 1`? Яка модель обрана за кросвалідацією?
3. Які оптимальні гіперпараметри отримано? Які помилки видно в матриці плутанини?
4. Навіщо відкладати тестові дані та виконувати масштабування й SMOTE тільки на train?

**Додаткове завдання.**

Реалізувати моделі **XGBoost, LightGBM та CatBoost.**

Вивести класифікаційний звіт та матрицю плутанини для кожної моделі.

Оцінити, зробити висновки про отримані метрики

In [ ]:
# Додаткове завдання: XGBoost, LightGBM, CatBoost
# Якщо пакети ще не встановлено, виконайте цю клітинку.
!pip -q install xgboost lightgbm catboost

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

extra_models = {
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1,
                             subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
                             random_state=42, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.1,
                               random_state=42, verbosity=-1),
    'CatBoost': CatBoostClassifier(iterations=100, depth=5, learning_rate=0.1,
                                  verbose=False, random_seed=42)
}

# У pipeline масштабування та SMOTE робимо лише на навчальних даних.
for name, model in extra_models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=42)), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    print(f'\n===== {name} =====')
    print(classification_report(y_test, pred, zero_division=0))
    cm = confusion_matrix(y_test, pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'Матриця плутанини: {name}')
    plt.xlabel('Передбачений клас'); plt.ylabel('Фактичний клас')
    plt.show()